# KG1 v73 — UNSLOTH SFT (REPLICA kienngx 0.86 baseline)

## CRITICO: receita alinhada com kienngx/nvidia-nemotron-training (Kaggle score 0.86 reproducivel)

**Config kienngx EXATA (verificada via OCTO Round 2A):**
- target_modules = "all-linear" (PEFT resolve em save para list safe)
- LoRA r=32, alpha=32 (ratio 1:1)
- lora_dropout=0.05
- lr=1e-4 (5x maior que our v73 previous 2e-5)
- gradient_accumulation=4
- max_length=2048 (raw answer column, nao CoT long)
- num_train_epochs=2
- optim='adamw_torch' (NAO adamw_8bit, Unsloth mais estavel)
- Dataset: train.csv OFICIAL Kaggle, sample(n=1200, seed=42)

**Score esperado: 0.86 +/- 0.01 (variance vLLM)**

**Por que trocar de huikang-16k para train.csv-1200:**
- kienngx prova que 1200 random samples > 16K (overfitting)
- Kaggle train.csv e distribuicao ORIGINAL dos puzzles (sem aug bias)
- seed=42 garante reproducibility
- Huikang 16k tem 68% categorias NAO-canonical + 659 IDs duplicados + 53.4% boxed malformado

**Memoria A100 40GB (huikang analysis):** μ=1 = 83GB teorico mas kienngx roda em RTX Pro 6000 Kaggle (96GB). Em A100 40GB com NF4 4bit: cabe com margin (estimado 30-35GB peak).

In [ ]:
# Cell 1: GPU diagnostic + anti-idle
import os, torch, subprocess, sys
r = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv', shell=True, capture_output=True, text=True)
print(r.stdout)
_lines = r.stdout.split(chr(10))
GPU_NAME = _lines[1].split(',')[0].strip() if len(_lines) > 1 else 'UNKNOWN'
print(f'GPU detected: {GPU_NAME}')
print(f'Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

# Anti-idle JS
from IPython.display import display, Javascript
display(Javascript("function ClickConnect(){document.querySelector('colab-connect-button').click()};setInterval(ClickConnect, 60000)"))

# Clone KG1 repo (OPCIONAL - nao usado no fluxo kienngx)
os.makedirs('/content', exist_ok=True)
if not os.path.exists('/content/kg1'):
    os.system('git clone https://github.com/FELIPEACASTRO/KG1.git /content/kg1 2>/dev/null || echo repo-missing')
sys.path.insert(0, '/content/kg1/src')
sys.path.insert(0, '/content')


In [ ]:
# Cell 2: Install Unsloth (notebook OFICIAL Unsloth para Nemotron-3-Nano-30B)
%%capture
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q --no-deps 'trl>=0.16' 'peft>=0.18.1' accelerate bitsandbytes
!pip install -q 'transformers>=4.55' liger-kernel datasets
!pip install -q hf_transfer
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
# Verify TRL version (max_length API requires 0.16+)
import trl
assert trl.__version__ >= '0.16', f'TRL {trl.__version__} < 0.16 (max_length kwarg requires 0.16+)'
print(f'TRL {trl.__version__} OK')


In [ ]:
# Cell 3: Drive mount + secrets (os ja importado em Cell 1)
from google.colab import drive, userdata
drive.mount('/content/drive')

try:
    HF_TOKEN = userdata.get('HF_KEY')
except Exception:
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = os.environ.get('HF_TOKEN', '')
        print('WARN: configure HF_KEY ou HF_TOKEN no Colab Secrets!')

assert HF_TOKEN and HF_TOKEN.startswith('hf_'), f'Invalid HF_TOKEN: {HF_TOKEN[:10]}'
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

CKPT_DIR = '/content/drive/MyDrive/kg1_v73_unsloth_moe'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoint dir: {CKPT_DIR}')


In [ ]:
# Cell 4: Load model Unsloth pre-quantizado (kienngx usa 2048 seq)
from unsloth import FastLanguageModel
import torch

MAX_SEQ = 2048  # kienngx: assistant contem raw answer, nao CoT longo

model, tok = FastLanguageModel.from_pretrained(
    model_name='unsloth/Nemotron-3-Nano-30B-A3B-bnb-4bit',
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    full_finetuning=False,
    token=HF_TOKEN,
)
print(f'Model loaded. MAX_SEQ: {MAX_SEQ}')


In [ ]:
# Cell 5: PEFT LoRA - REPLICA EXATA kienngx
# target_modules='all-linear': PEFT resolve em save_pretrained() para list explicita
# NemotronH NAO tem gate_proj/x_proj => safe para submission gate
# Lista resolvida: ['in_proj','out_proj','q_proj','k_proj','v_proj','o_proj','up_proj','down_proj','lm_head']

model = FastLanguageModel.get_peft_model(
    model,
    r=32,                          # kienngx: 32 (era 16 no nosso)
    lora_alpha=32,                 # kienngx: 32 (ratio 1:1)
    lora_dropout=0.05,             # kienngx: 0.05 (era 0.0)
    bias='none',
    use_rslora=False,
    use_dora=False,
    target_modules='all-linear',   # kienngx: string literal, PEFT resolve
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

model.print_trainable_parameters()
# Esperado: ~120-150M trainable (all-linear com r=32)
# Submission gate check: target_modules resolve para list contendo 'in_proj', sem 'gate_proj' OK


In [ ]:
# Cell 6: Dataset train.csv OFICIAL Kaggle (replica kienngx: 1200 samples random seed=42)
import polars as pl
import pandas as pd
from datasets import Dataset

# Baixar train.csv oficial do Kaggle (pode ser via Drive ou kaggle API)
TRAIN_CSV = '/content/drive/MyDrive/kg1_train.csv'
if not os.path.exists(TRAIN_CSV):
    # Fallback: download via kaggle API
    os.makedirs('/root/.kaggle', exist_ok=True)
    import shutil, json as _json
    kaggle_json_drive = '/content/drive/MyDrive/.kaggle/kaggle.json'
    if os.path.exists(kaggle_json_drive):
        shutil.copy(kaggle_json_drive, '/root/.kaggle/kaggle.json')
    else:
        try:
            from google.colab import userdata
            kaggle_user = userdata.get('KAGGLE_USERNAME')
            kaggle_key = userdata.get('KAGGLE_KEY')
            with open('/root/.kaggle/kaggle.json', 'w') as f:
                _json.dump({'username': kaggle_user, 'key': kaggle_key}, f)
        except Exception as e:
            raise RuntimeError(f'Configure KAGGLE_USERNAME/KEY no Colab Secrets ou kaggle.json no Drive: {e}')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    os.system('kaggle competitions download -c nvidia-nemotron-model-reasoning-challenge -f train.csv -p /content/')
    os.system('unzip -o /content/train.csv.zip -d /content/ 2>/dev/null || true')
    TRAIN_CSV = '/content/train.csv'

print(f'Loading train.csv from {TRAIN_CSV}')
df_full = pd.read_csv(TRAIN_CSV)
print(f'Train.csv full: {len(df_full)} rows | columns: {list(df_full.columns)}')

# kienngx EXATO: sample 1200 seed=42
SUBSAMPLE_SIZE = 1200
df = df_full.sample(n=SUBSAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f'Subsampled: {len(df)} rows')

# Detect prompt column
PROMPT_COL = 'prompt' if 'prompt' in df.columns else 'problem'
ANSWER_COL = 'answer' if 'answer' in df.columns else 'solution'
assert PROMPT_COL in df.columns, f'No prompt column in {df.columns}'
assert ANSWER_COL in df.columns, f'No answer column in {df.columns}'

# Prompt template kienngx EXATO
PROMPT_SUFFIX = chr(10) + 'Put your final answer inside \\boxed{}.'

def format_kienngx(row):
    user_msg = str(row[PROMPT_COL]) + PROMPT_SUFFIX
    assistant_msg = str(row[ANSWER_COL])  # raw answer column (contem CoT + boxed)
    messages = [
        {'role': 'user', 'content': user_msg},
        {'role': 'assistant', 'content': assistant_msg},
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {'text': text}

ds_train = Dataset.from_pandas(df).map(format_kienngx, num_proc=2, remove_columns=list(df.columns))
print(f'Formatted: {len(ds_train)} examples')
print('Sample text (first 400 chars):')
print(ds_train[0]['text'][:400])


In [ ]:
# Cell 7: SFT Training - REPLICA EXATA kienngx (TRL 0.16+)
from trl import SFTTrainer, SFTConfig
import threading, time, trl
print(f'Using TRL {trl.__version__}')
assert trl.__version__ >= '0.16', f'TRL {trl.__version__} < 0.16 required for max_length'

args = SFTConfig(
    output_dir=CKPT_DIR,
    # === kienngx EXATO ===
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,          # kienngx: 4 (era 16)
    num_train_epochs=2,
    learning_rate=1e-4,                     # kienngx: 1e-4 (era 2e-5)
    warmup_ratio=0.1,                       # kienngx: 0.1 (era 0.03)
    lr_scheduler_type='cosine',             # kienngx: cosine (era linear)
    logging_steps=10,
    save_steps=100,                         # sample 1200 * 2 epochs = 600 steps total
    save_total_limit=3,
    bf16=True,
    optim='adamw_torch',                    # kienngx: adamw_torch (era adamw_8bit)
    max_length=MAX_SEQ,                     # 2048 kienngx
    dataset_text_field='text',
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},  # kienngx
    max_grad_norm=1.0,                      # kienngx
    report_to='none',
    push_to_hub=False,
    seed=42,
    remove_unused_columns=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds_train,
    args=args,
    processing_class=tok,
)

# Memory monitoring (thread background)
def monitor_mem():
    while True:
        try:
            m = torch.cuda.memory_allocated()/1e9
            p = torch.cuda.max_memory_allocated()/1e9
            print(f'[MEM] {m:.1f}GB / peak {p:.1f}GB')
        except Exception:
            pass
        time.sleep(300)
threading.Thread(target=monitor_mem, daemon=True).start()

# Resume se existir
resume = None
if os.path.exists(CKPT_DIR):
    ckpts = [d for d in os.listdir(CKPT_DIR) if d.startswith('checkpoint-')]
    if ckpts:
        resume = True
        print(f'Resuming from checkpoint')

stats = trainer.train(resume_from_checkpoint=resume)
print(f'Training done: {stats}')
print(f'Final loss: {stats.training_loss:.4f} (esperado: 0.5-1.5 kienngx)')


In [ ]:
# Cell 8: Save final adapter to Drive + upload to HF
FINAL_DIR = f'{CKPT_DIR}/final_adapter'
trainer.save_model(FINAL_DIR)
tok.save_pretrained(FINAL_DIR)
print(f'Final adapter saved: {FINAL_DIR}')

# Upload to HF
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
REPO_ID = 'felipesp1983/kg1-nemotron-lora-v73-unsloth-moe'
api.create_repo(REPO_ID, private=True, exist_ok=True)
api.upload_folder(folder_path=FINAL_DIR, repo_id=REPO_ID, path_in_repo='final')
print(f'Uploaded to HF: {REPO_ID}')